In [ ]:
!pip -q install cohere pandas openpyxl

In [3]:
pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 43.3 MB/s eta 0:00:00


In [4]:
import os
import json
import pandas as pd
from getpass import getpass
from google.colab import files


print("Upload your Excel file containing student performance data.")
uploaded = files.upload()

INPUT_FILE = next(iter(uploaded)) if uploaded else ""

if not INPUT_FILE:
    INPUT_FILE = "student_performance.xlsx"

if not os.path.exists(INPUT_FILE):
    sample = pd.DataFrame([
        [1, "Ananya", 85, 90, 95, 96, "Loops", "Functions"],
        [2, "Rahul", 65, 42, 55, 68, "Functions, Loops", "Variables"],
        [3, "Priya", 75, 80, 40, 92, "Functions", "Loops"],
        [4, "Arjun", 88, 82, 90, 95, "None", "All topics"],
        [5, "Meera", 50, 45, 60, 72, "Loops, Functions", "Variables"],
        [6, "Karthik", 92, 88, 95, 98, "None", "All topics"],
        [7, "Divya", 70, 55, 45, 80, "Functions", "Variables"],
        [8, "Vikram", 60, 65, 70, 85, "Loops", "Variables"]
    ], columns=[
        "Student_ID", "Name", "Test1", "Test2",
        "Assignment", "Attendance",
        "Topics_Weak", "Topics_Strong"
    ])

    sample.to_excel(INPUT_FILE, index=False)
    print("Sample Excel file created.")

# Read Excel data
df = pd.read_excel(INPUT_FILE)
df.columns = df.columns.str.strip()

required = {
    "Student_ID", "Name", "Test1", "Test2",
    "Assignment", "Attendance",
    "Topics_Weak", "Topics_Strong"
}

missing = required - set(df.columns)

if missing:
    raise ValueError(
        f"Missing required Excel columns: {sorted(missing)}"
    )

for col in ["Test1", "Test2", "Assignment", "Attendance"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Student_ID", "Name", "Test1", "Test2"])

print("\nExcel data loaded successfully.")
print(df.head())




def performance_overview():
    data = df.copy()

    data["Average"] = data[["Test1", "Test2"]].mean(axis=1)
    data["Change"] = data["Test2"] - data["Test1"]

    return {
        "class_average": round(float(data["Average"].mean()), 2),
        "student_count": len(data),
        "improving_students": data.loc[
            data["Change"] > 0, "Name"
        ].tolist(),
        "declining_students": data.loc[
            data["Change"] < 0, "Name"
        ].tolist(),
        "student_results": data[
            ["Name", "Test1", "Test2", "Average", "Change"]
        ].round(2).to_dict(orient="records")
    }


# TOOL 2: Investigate attendance and scores

def investigate_attendance():
    data = df.dropna(subset=["Attendance", "Test2"])

    correlation = None

    if (
        len(data) >= 3
        and data["Attendance"].nunique() > 1
        and data["Test2"].nunique() > 1
    ):
        correlation = round(
            float(data["Attendance"].corr(data["Test2"])), 3
        )

    return {
        "attendance_score_correlation": correlation,
        "student_data": data[
            ["Name", "Attendance", "Test2"]
        ].to_dict(orient="records"),
        "caution": "Correlation does not prove causation."
    }


# TOOL 3: Investigate assignment performance

def investigate_assignments():
    data = df.dropna(subset=["Assignment", "Test2"])

    correlation = None

    if (
        len(data) >= 3
        and data["Assignment"].nunique() > 1
        and data["Test2"].nunique() > 1
    ):
        correlation = round(
            float(data["Assignment"].corr(data["Test2"])), 3
        )

    return {
        "assignment_score_correlation": correlation,
        "student_data": data[
            ["Name", "Assignment", "Test2"]
        ].to_dict(orient="records"),
        "caution": "Small samples limit reliable conclusions."
    }


# TOOL 4: Investigate one student

def investigate_student(student_name):
    match = df[
        df["Name"].astype(str).str.casefold()
        == student_name.strip().casefold()
    ]

    if match.empty:
        return {"error": f"Student not found: {student_name}"}

    row = match.iloc[0]

    return {
        "student": str(row["Name"]),
        "test1": float(row["Test1"]),
        "test2": float(row["Test2"]),
        "change": round(float(row["Test2"] - row["Test1"]), 2),
        "assignment": (
            None if pd.isna(row["Assignment"])
            else float(row["Assignment"])
        ),
        "attendance": (
            None if pd.isna(row["Attendance"])
            else float(row["Attendance"])
        ),
        "weak_topics": str(row["Topics_Weak"]),
        "strong_topics": str(row["Topics_Strong"])
    }


# TOOL 5: Generate draft support plans

def generate_recommendations():
    plans = []

    for _, row in df.iterrows():
        change = row["Test2"] - row["Test1"]
        evidence = []
        actions = []

        if change < 0:
            evidence.append("Test score declined")
            actions.append("Review errors in the latest test")

        if pd.notna(row["Attendance"]) and row["Attendance"] < 75:
            evidence.append("Attendance below 75%")
            actions.append("Discuss attendance barriers")

        if pd.notna(row["Assignment"]) and row["Assignment"] < 50:
            evidence.append("Assignment score below 50")
            actions.append("Provide guided assignment practice")

        weak = str(row["Topics_Weak"]).strip()

        if weak.lower() not in ["", "none", "nan"]:
            actions.append(f"Revise weak topics: {weak}")

        if not actions:
            actions.append("Continue learning plan and monitor progress")

        plans.append({
            "Student_ID": row["Student_ID"],
            "Name": row["Name"],
            "Test_Change": change,
            "Evidence": "; ".join(evidence) if evidence else
                        "No listed risk indicator",
            "Weak_Topics": weak,
            "Suggested_Actions": "; ".join(actions),
            "Teacher_Review": "Pending"
        })

    return plans


# Register executable functions

tool_functions = {
    "performance_overview": performance_overview,
    "investigate_attendance": investigate_attendance,
    "investigate_assignments": investigate_assignments,
    "investigate_student": investigate_student,
    "generate_recommendations": generate_recommendations
}



tools = [
    {
        "type": "function",
        "function": {
            "name": "performance_overview",
            "description": "Analyze class averages, individual scores, and declining or improving students.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "investigate_attendance",
            "description": "Analyze attendance and its association with test scores.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "investigate_assignments",
            "description": "Analyze assignment scores and their association with test results.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "investigate_student",
            "description": "Retrieve an individual student's scores, attendance, and topics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_name": {
                        "type": "string",
                        "description": "Exact student name from the Excel file"
                    }
                },
                "required": ["student_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_recommendations",
            "description": "Generate draft, evidence-based support plans for all students.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]




def execute_tool(name, arguments):
    if name not in tool_functions:
        return {"error": "Tool not found"}

    if name == "investigate_student":
        student_name = str(arguments.get("student_name", ""))
        if not student_name.strip():
            return {"error": "Student name is required"}
        return investigate_student(student_name)

    return tool_functions[name]()




query = """
Analyze the class performance and identify declining students.
Investigate attendance and assignment scores in relation to test
performance. Examine students who need attention, recommend
personalized learning interventions, and explain the evidence.
"""





import cohere

api_key = getpass("Enter your Cohere API key: ")
client = cohere.ClientV2(api_key=api_key)

messages = [
    {
        "role": "system",
        "content": """
You are an intelligent teaching assistant.

Understand the teacher's request and select relevant tools.
For complex questions, use multiple tools.
Use tool results as evidence, not assumptions.
Do not claim correlation proves causation.
Do not invent student data.
Provide practical recommendations for teacher review.
Call additional tools when more evidence is required.
Do not reveal private information beyond the supplied dataset.
"""
    },
    {
        "role": "user",
        "content": query
    }
]




tool_log = []
final_answer = None

for step in range(10):

    response = client.chat(
        model="command-a-03-2025",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    assistant_message = response.message
    messages.append(assistant_message)

    calls = assistant_message.tool_calls or []

    # No more tools requested: the LLM has prepared its answer
    if not calls:
        final_answer = "".join(
            item.text
            for item in (assistant_message.content or [])
            if getattr(item, "text", None)
        )
        break

    # Execute each selected tool
    for call in calls:
        name = call.function.name
        arguments = call.function.arguments or {}

        if isinstance(arguments, str):
            arguments = json.loads(arguments)

        print(f"\nAgent selected tool: {name}")
        print("Arguments:", arguments)

        result = execute_tool(name, arguments)

        tool_log.append({
            "Step": step + 1,
            "Tool": name,
            "Arguments": json.dumps(arguments),
            "Result": json.dumps(result, default=str)
        })

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result, default=str)
        })

if final_answer is None:
    final_answer = (
        "The agent reached the maximum number of iterations. "
        "Review the tool log and try a more focused question."
    )

print("\n" + "=" * 60)
print("TEACHING ASSISTANT AGENT RESPONSE")
print("=" * 60)
print(final_answer)




overview = performance_overview()
attendance = investigate_attendance()
assignments = investigate_assignments()
plans = generate_recommendations()

with pd.ExcelWriter(
    "teaching_assistant_report.xlsx",
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer, sheet_name="Student_Data", index=False
    )

    pd.DataFrame(
        overview["student_results"]
    ).to_excel(
        writer, sheet_name="Performance", index=False
    )

    pd.DataFrame(
        [attendance]
    ).to_excel(
        writer, sheet_name="Attendance_Analysis", index=False
    )

    pd.DataFrame(
        [assignments]
    ).to_excel(
        writer, sheet_name="Assignment_Analysis", index=False
    )

    pd.DataFrame(
        plans
    ).to_excel(
        writer, sheet_name="Support_Plans", index=False
    )

    pd.DataFrame(
        tool_log
    ).to_excel(
        writer, sheet_name="Agent_Tool_Log", index=False
    )

    pd.DataFrame([{
        "Teacher_Query": query,
        "Agent_Answer": final_answer
    }]).to_excel(
        writer, sheet_name="Agent_Answer", index=False
    )

print("\nExcel report created successfully.")

files.download("teaching_assistant_report.xlsx")

Upload your Excel file containing student performance data.



Excel data loaded successfully.
   Student_ID    Name  Test1  Test2  Assignment  Attendance       Topics_Weak  \
0           1  Ananya     85     90          95          96             Loops   
1           2   Rahul     65     42          55          68  Functions, Loops   
2           3   Priya     75     80          40          92         Functions   
3           4   Arjun     88     82          90          95               NaN   
4           5   Meera     50     45          60          72  Loops, Functions   

  Topics_Strong  
0     Functions  
1     Variables  
2         Loops  
3    All topics  
4     Variables  
Enter your Cohere API key: ··········

Agent selected tool: performance_overview
Arguments: {}

Agent selected tool: investigate_attendance
Arguments: {}

Agent selected tool: investigate_assignments
Arguments: {}

Agent selected tool: generate_recommendations
Arguments: {}

TEACHING ASSISTANT AGENT RESPONSE
## Class Performance
The class average is 70.75. The improving

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>